# DenseNet121 scratch — kiểm định chất lượng Cashew_dataV06

**Mục tiêu:** tìm lỗi cấu trúc, ảnh hỏng, ảnh trùng/na ná giữa split hoặc khác nhãn, lệch phân phối ảnh và thiếu đại diện nguồn `real_`; tạo danh sách ảnh cần chuyên gia xem; dùng DenseNet121 `weights=None` để đo khả năng học và các lỗi bền vững trên Validation. Notebook này không tự kết luận nhãn sai và không tự xóa/đổi nhãn.

**Cách chạy:** chạy tuần tự tất cả cell. Phần audit tĩnh chạy trước và không cần GPU. Phần scratch probe ở cuối mặc định chạy 2 seed, 30 epoch/seed; có thể đặt `RUN_SCRATCH_PROBE=False` để chỉ lấy báo cáo audit. Cần `numpy`, `pandas`, `Pillow`, `scipy`, `matplotlib`, `scikit-learn`, `tensorflow` và Jupyter/IPython. Chỉnh `DATASET_ROOT_OVERRIDE` / `ARCHIVE_OVERRIDE` ở cell cấu hình nếu cần. Colab sẽ mount Drive và giải nén ZIP V06 khi chưa có thư mục dữ liệu. Kết quả lưu trong thư mục `EXP-DENSENET121-DATASET-AUDIT-001` riêng.

**Phiên bản dữ liệu:** theo notebook 007/A_4: Train 4.825, Val 1.395, Test 687. Hai tệp `val/leaf_miner/leaf_miner_1056.jpg` và `leaf_miner_1091.jpg` không có trong ZIP mới; nếu một bản cũ đưa chúng trở lại, notebook vẫn audit chúng và loại khỏi phép thử mô hình. Fingerprint từ hash nội dung sẽ chỉ ra khác biệt với A_4. **Test chỉ dùng để kiểm tra cấu trúc và ảnh trùng**, không dùng để chọn checkpoint hay báo cáo độ chính xác mô hình.

**Đọc kết quả:** `audit_summary.json` và các CSV là bằng chứng trực tiếp; pHash/độ nét/độ sáng chỉ tạo *ứng viên cần xem*. `review_contact_sheet.png`, `near_duplicate_pairs.csv`, `validation_review_queue.csv` là danh sách rà nhãn/nguồn. Macro-F1 và gap phụ thuộc recipe/seed; spike Validation có thể liên quan BatchNorm/LR, không đủ để kết luận dataset kém. Thiếu ID cây/lá/buổi chụp thì không thể xác nhận split độc lập theo nhóm.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import hashlib, json, math, os, random, shutil, zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageDraw
from scipy.fft import dctn
from scipy.ndimage import laplace
from scipy.spatial.distance import cdist
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
try:
    from IPython.display import display
except ImportError:
    display = print

CLASSES = ["anthracnose", "healthy", "leaf_miner", "not_cashew_leaf", "red_rust"]
SPLITS = ["train", "val", "test"]
EXPECTED = {
    "anthracnose": [948, 299, 122], "healthy": [806, 213, 118],
    "leaf_miner": [893, 246, 132], "not_cashew_leaf": [1101, 314, 157],
    "red_rust": [1077, 323, 158],
}
KNOWN_OLD_VAL_COPIES = {
    "val/leaf_miner/leaf_miner_1056.jpg",
    "val/leaf_miner/leaf_miner_1091.jpg",
}
KNOWN_A4_FINGERPRINTS = {
    "train": "3c25479ef5208b94f10de638ace73b7f6f850e3a78664a52262d067e1b9e53c8",
    "val": "3d7672f83c20797b5a569faa39be7fa1f8976fcf6b8c0c8d862eff7004fdb442",
    "test": "eeb8b3079cce49b792ec50c375cc2d2ab4748fa6a435c3be29ebedea4caef735",
}

DATASET_ROOT_OVERRIDE = None  # e.g. Path('/kaggle/input/.../CashewData_Split')
ARCHIVE_OVERRIDE = None       # e.g. Path('/content/drive/MyDrive/.../CashewData_Split (7).zip')
OUTPUT_ROOT_OVERRIDE = None
RUN_SCRATCH_PROBE = True
REQUIRE_A4_FINGERPRINT = True  # set False only when intentionally auditing another revision
SEEDS = [42, 7777]
EPOCHS = 30
BATCH_SIZE = 32
IMAGE_SIZE = 224
PEAK_LR = 3e-4
WARMUP_EPOCHS = 3
MIN_LR = 1e-6
PHASH_HAMMING_MAX = 6   # candidate only, not a duplicate verdict
NEAR_PAIR_LIMIT = 20000  # cap report size; nearest candidates retained first

if os.name == "nt":
    default_root = Path(r"D:\Study\KhoaLuanTotNghiep\Dataset\CashewData_Split")
    default_output = Path(r"D:\Study\KhoaLuanTotNghiep\Resource\training_results\current_dataset")
elif Path('/kaggle/input').exists():
    candidates = sorted(p for p in Path('/kaggle/input').rglob('CashewData_Split')
                        if p.is_dir() and all((p / s).is_dir() for s in SPLITS))
    if DATASET_ROOT_OVERRIDE is None and len(candidates) != 1:
        raise RuntimeError(f"Set DATASET_ROOT_OVERRIDE; candidates: {candidates}")
    default_root = candidates[0] if candidates else Path('/kaggle/input/CashewData_Split')
    default_output = Path('/kaggle/working')
else:
    default_root = Path('/content/CashewData_Split')
    default_output = Path('/content')

DATASET_ROOT = Path(DATASET_ROOT_OVERRIDE or default_root)
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE or default_output)
OUT = OUTPUT_ROOT / 'EXP-DENSENET121-DATASET-AUDIT-001'
OUT.mkdir(parents=True, exist_ok=True)

if not all((DATASET_ROOT / s).is_dir() for s in SPLITS) and Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    archive = Path(ARCHIVE_OVERRIDE or
        '/content/drive/MyDrive/Cashew_Leaf_model_result/Dataset/CashewData_Split (7).zip')
    if not archive.is_file():
        raise FileNotFoundError(f"Dataset ZIP not found: {archive}")
    with zipfile.ZipFile(archive) as zf:
        target_root = Path('/content').resolve()
        for member in zf.infolist():
            target = (target_root / member.filename).resolve()
            if target != target_root and target_root not in target.parents:
                raise ValueError(f"Unsafe ZIP member: {member.filename}")
        zf.extractall(target_root)
if not all((DATASET_ROOT / s).is_dir() for s in SPLITS):
    raise FileNotFoundError(f"Missing train/val/test below {DATASET_ROOT}")
print('Dataset:', DATASET_ROOT, '\nOutput:', OUT, '\nScratch probe:', RUN_SCRATCH_PROBE)


## 1. Kiểm tra từng tệp và khóa danh tính dataset

Tính SHA-256 byte và SHA-256 pixel RGB sau EXIF transpose, đọc toàn ảnh để phát hiện lỗi decode. Ghi kích thước, độ sáng, tỷ lệ pixel quá tối/quá sáng và phương sai Laplacian trên bản thu nhỏ. Các phép đo chất lượng ảnh chỉ có ý nghĩa **tương đối trong từng lớp**; không tự động loại ảnh theo một ngưỡng chung.


In [ ]:
VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.gif'}
rows, invalid, unknown = [], [], []
class_folder_issues = []

def file_sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def phash64(image):
    gray = np.asarray(ImageOps.grayscale(image).resize((32, 32), Image.Resampling.LANCZOS), dtype=np.float32)
    freq = dctn(gray, norm='ortho')[:8, :8]
    median = np.median(freq.flat[1:])
    return ''.join('1' if v > median else '0' for v in freq.ravel())

for split in SPLITS:
    found = sorted(p.name for p in (DATASET_ROOT / split).iterdir() if p.is_dir())
    if found != sorted(CLASSES):
        class_folder_issues.append({'split': split, 'expected': CLASSES, 'found': found})
    for label, cls in enumerate(CLASSES):
        folder = DATASET_ROOT / split / cls
        if not folder.is_dir():
            continue
        for path in sorted(folder.rglob('*')):
            if not path.is_file():
                continue
            rel = path.relative_to(DATASET_ROOT).as_posix()
            if path.suffix.lower() not in VALID_EXT:
                unknown.append({'relative_path': rel, 'reason': 'unsupported extension'})
                continue
            try:
                with Image.open(path) as img:
                    rgb = ImageOps.exif_transpose(img).convert('RGB')
                    rgb.load()
                small = np.asarray(rgb.resize((256, 256), Image.Resampling.BILINEAR), dtype=np.uint8)
                gray = np.asarray(ImageOps.grayscale(rgb).resize((256, 256), Image.Resampling.BILINEAR), dtype=np.float32)
                rows.append({
                    'split': split, 'class': cls, 'label': label,
                    'relative_path': rel, 'path': str(path),
                    'is_real_filename': path.name.casefold().startswith('real_'),
                    'sha256': file_sha256(path),
                    'pixel_sha256': hashlib.sha256(
                        f'{rgb.width}x{rgb.height}:'.encode() + rgb.tobytes()).hexdigest(),
                    'width': rgb.width, 'height': rgb.height, 'aspect_ratio': rgb.width / rgb.height,
                    'bytes': path.stat().st_size,
                    'brightness': float(gray.mean() / 255),
                    'contrast': float(gray.std() / 255),
                    'blur_laplacian_var': float(laplace(gray).var()),
                    'dark_fraction': float(np.mean(gray < 10)),
                    'bright_fraction': float(np.mean(gray > 245)),
                    'mean_red': float(small[:, :, 0].mean() / 255),
                    'mean_green': float(small[:, :, 1].mean() / 255),
                    'mean_blue': float(small[:, :, 2].mean() / 255),
                    'phash_bits': phash64(rgb),
                })
            except Exception as exc:
                invalid.append({'relative_path': rel, 'error': repr(exc)})
    print(split, 'scanned')

manifest = pd.DataFrame(rows).sort_values(['split', 'class', 'relative_path']).reset_index(drop=True)
if manifest.empty:
    raise RuntimeError('No decodable images found')
pd.DataFrame(invalid, columns=['relative_path', 'error']).to_csv(OUT / 'invalid_images.csv', index=False)
pd.DataFrame(unknown, columns=['relative_path', 'reason']).to_csv(OUT / 'unknown_files.csv', index=False)
manifest.to_csv(OUT / 'dataset_manifest.csv', index=False)
counts = pd.crosstab(manifest['class'], manifest['split']).reindex(index=CLASSES, columns=SPLITS, fill_value=0)
display(counts)
expected = pd.DataFrame.from_dict(EXPECTED, orient='index', columns=SPLITS).reindex(CLASSES)
count_diff = counts - expected
count_diff.to_csv(OUT / 'count_difference_vs_v06.csv')
fingerprints = {}
for split in SPLITS:
    frame = manifest[manifest['split'].eq(split)].sort_values('relative_path')
    records = frame[['relative_path', 'label', 'sha256']].to_dict('records')
    payload = json.dumps(records, sort_keys=True, separators=(',', ':')).encode()
    fingerprints[split] = hashlib.sha256(payload).hexdigest()
summary = {
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'dataset_root': str(DATASET_ROOT), 'total_decodable': len(manifest),
    'invalid_count': len(invalid), 'unknown_count': len(unknown),
    'class_folder_issues': class_folder_issues,
    'counts_match_v06': bool((count_diff == 0).all().all()),
    'fingerprints': fingerprints,
    'matches_known_a4_fingerprints': {s: fingerprints[s] == KNOWN_A4_FINGERPRINTS[s] for s in SPLITS},
    'known_old_val_copies_present': sorted(KNOWN_OLD_VAL_COPIES & set(manifest.relative_path)),
}
(OUT / 'audit_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
display(pd.DataFrame([summary]).drop(columns=['class_folder_issues', 'fingerprints']))
print('Fingerprints:', fingerprints)
if invalid or class_folder_issues:
    print('STOP before training: inspect invalid_images.csv / class folders.')


## 2. Trùng lặp và khả năng rò rỉ split

SHA trùng là bằng chứng tệp/pixel trùng. pHash cách Hamming ≤6 chỉ là ứng viên gần trùng; ảnh lá khác nhau có thể rất giống và ảnh crop/biến đổi mạnh có thể lọt qua. `mad_64_gray` là độ lệch trung bình ảnh xám 64×64 đã scale về [0,1]; giá trị ≤0,01 giúp ưu tiên xem, vẫn không thay thế kiểm tra ảnh gốc. Báo cáo riêng cặp giữa split/khác nhãn và cặp cùng split/cùng lớp để rà độ đa dạng. Phép lọc này không bất biến với xoay/crop lớn. Test không được đưa vào huấn luyện.


In [ ]:
exact_rows = []
for key in ['sha256', 'pixel_sha256']:
    for digest, group in manifest.groupby(key):
        if len(group) < 2:
            continue
        items = group[['relative_path', 'split', 'class']].to_dict('records')
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                exact_rows.append({'type': key, 'digest': digest, 'a': items[i]['relative_path'],
                                   'b': items[j]['relative_path'], 'a_split': items[i]['split'],
                                   'b_split': items[j]['split'], 'a_class': items[i]['class'],
                                   'b_class': items[j]['class']})
exact = pd.DataFrame(exact_rows, columns=['type', 'digest', 'a', 'b', 'a_split', 'b_split', 'a_class', 'b_class'])
exact.to_csv(OUT / 'exact_duplicate_pairs.csv', index=False)

bits = np.array([[c == '1' for c in s] for s in manifest.phash_bits], dtype=np.uint8)
near, within = [], []
# 128 x N distance blocks keep memory bounded. Consider only different splits or labels.
for start in range(0, len(manifest), 128):
    stop = min(start + 128, len(manifest))
    distances = cdist(bits[start:stop], bits, metric='hamming') * 64
    for local_i, global_i in enumerate(range(start, stop)):
        candidates = np.flatnonzero((distances[local_i] <= PHASH_HAMMING_MAX) &
                                    (np.arange(len(manifest)) > global_i))
        for j in candidates:
            a, b = manifest.iloc[global_i], manifest.iloc[j]
            target = within if a['split'] == b['split'] and a['class'] == b['class'] else near
            target.append({'a': a.relative_path, 'b': b.relative_path,
                         'a_split': a['split'], 'b_split': b['split'],
                         'a_class': a['class'], 'b_class': b['class'],
                         'hamming': int(distances[local_i, j]),
                         'same_pixel': bool(a.pixel_sha256 == b.pixel_sha256)})
near_df = pd.DataFrame(near, columns=['a', 'b', 'a_split', 'b_split', 'a_class', 'b_class', 'hamming', 'same_pixel'])
near_df = near_df.sort_values(['hamming', 'a', 'b']).head(NEAR_PAIR_LIMIT)
gray_cache = {}
def gray64(rel):
    if rel not in gray_cache:
        with Image.open(DATASET_ROOT / rel) as image:
            image = ImageOps.exif_transpose(image).convert('L').resize((64, 64), Image.Resampling.BILINEAR)
            gray_cache[rel] = np.asarray(image, dtype=np.float32) / 255
    return gray_cache[rel]
near_df['mad_64_gray'] = [float(np.mean(np.abs(gray64(a) - gray64(b))))
                          for a, b in zip(near_df.a, near_df.b)]
near_df['thumbnail_match_candidate'] = near_df.mad_64_gray <= .01
near_df = near_df.sort_values(['mad_64_gray', 'hamming', 'a', 'b'])
near_df.to_csv(OUT / 'near_duplicate_pairs.csv', index=False)
within_df = pd.DataFrame(within, columns=['a', 'b', 'a_split', 'b_split', 'a_class', 'b_class', 'hamming', 'same_pixel'])
within_df = within_df.sort_values(['hamming', 'a', 'b']).head(5000)
within_df.to_csv(OUT / 'within_split_near_pairs.csv', index=False)
summary.update({
    'exact_duplicate_pairs': int(len(exact)),
    'cross_split_or_cross_label_phash_candidates': int(len(near_df)),
    'within_split_same_class_phash_candidates_retained': int(len(within_df)),
    'thumbnail_match_candidates': near_df.loc[near_df.thumbnail_match_candidate,
                                               ['a', 'b', 'mad_64_gray']].to_dict('records'),
})
(OUT / 'audit_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print('Exact duplicate pairs:', len(exact), '| Cross-split / cross-label near candidates:', len(near_df),
      '| Within-split same-class candidates retained:', len(within_df))
if len(near_df): display(near_df.head(20))


## 3. Phân phối và danh sách ảnh cần xem

So chênh lệch trung bình chuẩn hóa (SMD) của các đặc trưng ảnh giữa Train và Validation theo từng lớp. `|SMD| > 0.5` là tín hiệu rà soát, không phải ngưỡng chất lượng được công nhận. Đếm riêng ảnh có tiền tố `real_` theo lớp/split; đây chỉ là dấu hiệu trong tên tệp, chưa xác minh nguồn chụp. Contact sheet chọn ảnh ở đuôi phân phối độ nét/độ sáng, ảnh gần trùng và vài ảnh ngẫu nhiên để chuyên gia kiểm tra thủ công.


In [ ]:
FEATURES = ['width', 'height', 'aspect_ratio', 'brightness', 'contrast',
            'blur_laplacian_var', 'dark_fraction', 'bright_fraction',
            'mean_red', 'mean_green', 'mean_blue']
real_counts = pd.crosstab([manifest['split'], manifest['class']], manifest.is_real_filename)
real_counts = real_counts.reindex(columns=[False, True], fill_value=0)
real_counts.columns = ['non_real_filename', 'real_filename']
real_counts.to_csv(OUT / 'real_filename_counts_by_split_class.csv')
display(real_counts)
summary['real_filename_counts_by_split'] = {s: int(manifest.loc[manifest['split'].eq(s), 'is_real_filename'].sum()) for s in SPLITS}
(OUT / 'audit_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
if summary['real_filename_counts_by_split']['val'] == 0:
    print('No real_ filename in Validation: Validation cannot measure the field-image subgroup.')

shift_rows = []
for cls in CLASSES:
    a = manifest[(manifest['split'] == 'train') & (manifest['class'] == cls)]
    b = manifest[(manifest['split'] == 'val') & (manifest['class'] == cls)]
    for feature in FEATURES:
        pooled_sd = math.sqrt((a[feature].var(ddof=1) + b[feature].var(ddof=1)) / 2)
        smd = (b[feature].mean() - a[feature].mean()) / pooled_sd if pooled_sd else 0.0
        shift_rows.append({'class': cls, 'feature': feature, 'train_mean': a[feature].mean(),
                           'val_mean': b[feature].mean(), 'smd': smd})
shift = pd.DataFrame(shift_rows).sort_values('smd', key=lambda s: s.abs(), ascending=False)
shift.to_csv(OUT / 'train_val_feature_shift.csv', index=False)
display(shift.head(20))

sizes = manifest.assign(image_size=manifest.width.astype(str) + 'x' + manifest.height.astype(str))
size_counts = sizes.groupby(['split', 'class', 'image_size']).size().rename('count').reset_index()
size_counts['fraction_within_class_split'] = size_counts['count'] / size_counts.groupby(
    ['split', 'class'])['count'].transform('sum')
size_counts.sort_values(['split', 'class', 'count'], ascending=[True, True, False]).to_csv(
    OUT / 'image_size_distribution.csv', index=False)
dominant_sizes = size_counts.sort_values('count', ascending=False).drop_duplicates(['split', 'class'])
display(dominant_sizes.sort_values(['split', 'class']))
print('Resolution can identify acquisition/source. Compare its class relationship across splits.')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
counts.plot.bar(ax=axes[0], title='Images by class and split')
shift.pivot(index='class', columns='feature', values='smd').reindex(CLASSES).plot.bar(
    ax=axes[1], title='Validation vs Train standardized mean difference', legend=False)
axes[1].axhline(.5, ls=':', color='red'); axes[1].axhline(-.5, ls=':', color='red')
fig.tight_layout(); fig.savefig(OUT / 'distribution_and_shift.png', dpi=150); plt.show()

review = []
for (split, cls), group in manifest.groupby(['split', 'class']):
    for feature, high in [('blur_laplacian_var', False), ('brightness', False),
                          ('brightness', True), ('dark_fraction', True), ('bright_fraction', True)]:
        for _, r in group.sort_values(feature, ascending=not high).head(3).iterrows():
            review.append({'relative_path': r.relative_path, 'reason': f'{feature}: ' + ('high' if high else 'low'),
                           'value': r[feature]})
    for _, r in group.sample(n=min(3, len(group)), random_state=42).iterrows():
        review.append({'relative_path': r.relative_path, 'reason': 'random control', 'value': np.nan})
for _, r in near_df.head(40).iterrows():
    review.extend([{'relative_path': r.a, 'reason': 'near pair ' + r.b, 'value': r.hamming},
                   {'relative_path': r.b, 'reason': 'near pair ' + r.a, 'value': r.hamming}])
review_df = pd.DataFrame(review).drop_duplicates('relative_path')
review_df.to_csv(OUT / 'image_review_candidates.csv', index=False)

def contact_sheet(paths, out_path, title, cols=6, thumb=(190, 150)):
    paths = list(paths)
    if not paths:
        return
    rows_n = math.ceil(len(paths) / cols)
    tile_w, tile_h = thumb[0] + 12, thumb[1] + 44
    canvas = Image.new('RGB', (cols * tile_w, rows_n * tile_h + 35), 'white')
    draw = ImageDraw.Draw(canvas); draw.text((8, 8), title, fill='black')
    for i, rel in enumerate(paths):
        try:
            with Image.open(DATASET_ROOT / rel) as im:
                pic = ImageOps.exif_transpose(im).convert('RGB')
                pic.thumbnail(thumb)
            x, y = (i % cols) * tile_w, (i // cols) * tile_h + 35
            canvas.paste(pic, (x + (thumb[0] - pic.width) // 2, y))
            draw.text((x + 4, y + thumb[1] + 2), rel[-30:], fill='black')
        except Exception as exc:
            print('Contact sheet skipped', rel, repr(exc))
    canvas.save(out_path)

contact_sheet(review_df.relative_path.head(120), OUT / 'review_contact_sheet.png',
              'Image quality and near-duplicate candidates')
pair_paths = [path for _, pair in near_df.head(36).iterrows() for path in (pair.a, pair.b)]
contact_sheet(pair_paths, OUT / 'near_pair_contact_sheet.png',
              'Near pairs: each consecutive two images form one candidate pair')
print('Review candidate images:', len(review_df))


## 4. DenseNet121 scratch probe

Kiến trúc như baseline: augmentation nhẹ, chia 255 đúng một lần, DenseNet121 `weights=None`, GAP, Dense(512), BatchNorm, Dropout(0.4), softmax. Lịch 3 epoch warmup + cosine 30 epoch được cố định trước; không dùng Test. Chọn checkpoint theo `val_loss`. Đo Train sạch trên 512 ảnh phân tầng ở **inference mode** mỗi epoch để tách train online khỏi gap thật. Hai seed giúp tìm ảnh Validation khó lặp lại, nhưng đây vẫn là phép thử chẩn đoán, chưa phải kết quả 5 seed của khóa luận.

Phần này cần GPU. Mặc định chỉ train nếu fingerprint khớp A_4; khi chủ ý dùng revision khác, đặt `REQUIRE_A4_FINGERPRINT=False` và ghi rõ phiên bản mới khi so sánh.


In [ ]:
if RUN_SCRATCH_PROBE:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    print('TensorFlow', tf.__version__, '| GPU', tf.config.list_physical_devices('GPU'))
    if invalid or unknown or class_folder_issues or not summary['counts_match_v06']:
        raise RuntimeError('Audit found structural differences; inspect CSVs before training.')
    if REQUIRE_A4_FINGERPRINT and not all(summary['matches_known_a4_fingerprints'].values()):
        raise RuntimeError('Dataset bytes differ from the known A_4 revision. Inspect audit_summary.json; '
                           'set REQUIRE_A4_FINGERPRINT=False only for an intentional new revision.')
    if not tf.config.list_physical_devices('GPU'):
        raise RuntimeError('Static audit completed. Select a GPU runtime for the DenseNet121 scratch probe.')
    if any((OUT / f'seed_{seed}').exists() for seed in SEEDS):
        raise FileExistsError('Probe output already exists. Choose a new OUTPUT_ROOT_OVERRIDE to keep prior evidence.')
    model_manifest = manifest[~manifest.relative_path.isin(KNOWN_OLD_VAL_COPIES)].copy()
    train_frame = model_manifest[model_manifest['split'] == 'train'].reset_index(drop=True)
    val_frame = model_manifest[model_manifest['split'] == 'val'].reset_index(drop=True)
    if train_frame.empty or val_frame.empty:
        raise RuntimeError('Train/Validation is empty')
    rng = np.random.default_rng(42)
    probe_parts = []
    for _, group in train_frame.groupby('class'):
        n = max(1, round(512 * len(group) / len(train_frame)))
        probe_parts.append(group.sample(n=min(n, len(group)), random_state=42))
    probe_frame = pd.concat(probe_parts).sort_values('relative_path').reset_index(drop=True)
    probe_frame[['relative_path', 'class']].to_csv(OUT / 'train_clean_probe_manifest.csv', index=False)

    def decode(path, label):
        data = tf.io.read_file(path)
        image = tf.io.decode_image(data, channels=3, expand_animations=False)
        image.set_shape((None, None, 3))
        image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
        return tf.cast(image, tf.float32), label

    def dataset(frame, seed=42, shuffle=False):
        ds = tf.data.Dataset.from_tensor_slices((frame.path.to_numpy(), frame.label.to_numpy(np.int32)))
        if shuffle:
            ds = ds.shuffle(len(frame), seed=seed, reshuffle_each_iteration=True)
        return ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    def build_model(seed):
        keras.utils.set_random_seed(seed)
        inputs = keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3))
        x = keras.Sequential([
            layers.RandomFlip('horizontal', seed=seed + 1),
            layers.RandomRotation(.04, seed=seed + 2),
            layers.RandomZoom((-.05, .05), seed=seed + 3),
            layers.RandomTranslation(.03, .03, seed=seed + 4),
            layers.RandomContrast(.08, seed=seed + 5),
        ], name='augmentation')(inputs)
        x = layers.Rescaling(1 / 255)(x)
        backbone = keras.applications.DenseNet121(include_top=False, weights=None,
                                                   input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
        x = backbone(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(512, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(.4, seed=seed + 6)(x)
        output = layers.Dense(len(CLASSES), activation='softmax')(x)
        return keras.Model(inputs, output, name=f'densenet121_scratch_dataset_probe_{seed}')

    def lr_at_epoch(epoch):
        if epoch < WARMUP_EPOCHS:
            return MIN_LR + (PEAK_LR - MIN_LR) * (epoch + 1) / WARMUP_EPOCHS
        progress = (epoch - WARMUP_EPOCHS + 1) / max(1, EPOCHS - WARMUP_EPOCHS)
        return MIN_LR + .5 * (PEAK_LR - MIN_LR) * (1 + math.cos(math.pi * progress))

    class CleanProbe(keras.callbacks.Callback):
        def __init__(self, ds):
            super().__init__(); self.ds = ds; self.rows = []
        def on_epoch_end(self, epoch, logs=None):
            loss, acc = self.model.evaluate(self.ds, verbose=0)
            self.rows.append({'epoch': epoch + 1, 'train_clean_probe_loss': float(loss),
                              'train_clean_probe_accuracy': float(acc),
                              'val_loss': float(logs['val_loss']), 'val_accuracy': float(logs['val_accuracy']),
                              'train_online_loss': float(logs['loss']),
                              'train_online_accuracy': float(logs['accuracy']),
                              'lr': lr_at_epoch(epoch)})

    probe_config = {'seeds': SEEDS, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
                    'image_size': IMAGE_SIZE, 'peak_lr': PEAK_LR,
                    'warmup_epochs': WARMUP_EPOCHS, 'min_lr': MIN_LR,
                    'checkpoint_selection': 'minimum_val_loss',
                    'weights': None, 'tensorflow': tf.__version__,
                    'dataset_fingerprints': summary['fingerprints'],
                    'test_used_for_model_metrics': False}
    (OUT / 'scratch_probe_config.json').write_text(json.dumps(probe_config, indent=2), encoding='utf-8')
    print('Train', len(train_frame), 'Val', len(val_frame), 'Clean probe', len(probe_frame))


In [ ]:
if RUN_SCRATCH_PROBE:
    val_ds = dataset(val_frame)
    probe_ds = dataset(probe_frame)
    all_val_predictions = []
    seed_summaries = []
    for seed in SEEDS:
        keras.backend.clear_session()
        keras.utils.set_random_seed(seed)
        seed_dir = OUT / f'seed_{seed}'
        seed_dir.mkdir(parents=True, exist_ok=False)
        model = build_model(seed)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=PEAK_LR),
                      loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        check = seed_dir / 'best_val_loss.weights.h5'
        clean = CleanProbe(probe_ds)
        callbacks = [keras.callbacks.LearningRateScheduler(lr_at_epoch, verbose=0),
                     keras.callbacks.ModelCheckpoint(check, monitor='val_loss', mode='min',
                                                     save_best_only=True, save_weights_only=True), clean]
        model.fit(dataset(train_frame, seed, shuffle=True), validation_data=val_ds,
                  epochs=EPOCHS, callbacks=callbacks, verbose=2)
        history = pd.DataFrame(clean.rows)
        history.to_csv(seed_dir / 'learning_curve.csv', index=False)
        best = history.sort_values(['val_loss', 'epoch']).iloc[0]
        model.load_weights(check)
        for split_name, frame in [('train', train_frame), ('val', val_frame)]:
            probabilities = model.predict(dataset(frame), verbose=0)
            if len(probabilities) != len(frame):
                raise RuntimeError(f'Prediction alignment failed: {split_name}')
            y = frame.label.to_numpy(np.int32)
            pred = probabilities.argmax(axis=1)
            selected = np.clip(probabilities[np.arange(len(y)), y], 1e-7, 1)
            result = frame[['relative_path', 'class', 'label']].copy()
            result['prediction'] = [CLASSES[i] for i in pred]
            result['correct'] = pred == y
            result['true_class_probability'] = selected
            result['predicted_confidence'] = probabilities.max(axis=1)
            result['cross_entropy'] = -np.log(selected)
            for idx, cls in enumerate(CLASSES): result['p_' + cls] = probabilities[:, idx]
            result.to_csv(seed_dir / f'{split_name}_predictions.csv', index=False)
            if split_name == 'val':
                report = classification_report(y, pred, target_names=CLASSES,
                                               labels=list(range(len(CLASSES))), output_dict=True,
                                               zero_division=0)
                (seed_dir / 'validation_classification_report.json').write_text(
                    json.dumps(report, indent=2), encoding='utf-8')
                pd.DataFrame(confusion_matrix(y, pred, labels=range(len(CLASSES))),
                             index=CLASSES, columns=CLASSES).to_csv(seed_dir / 'validation_confusion.csv')
                all_val_predictions.append(result.assign(seed=seed))
                val_acc = accuracy_score(y, pred)
                val_f1 = f1_score(y, pred, average='macro')
            else:
                train_acc = accuracy_score(y, pred)
                train_f1 = f1_score(y, pred, average='macro')
        seed_summaries.append({'seed': seed, 'best_epoch': int(best.epoch),
                               'best_val_loss': float(best.val_loss),
                               'train_clean_accuracy': train_acc, 'val_accuracy': val_acc,
                               'train_clean_macro_f1': train_f1, 'val_macro_f1': val_f1,
                               'clean_accuracy_gap_pp': 100 * (train_acc - val_acc)})
        print(seed_summaries[-1])
        del model
    summary_df = pd.DataFrame(seed_summaries)
    summary_df.to_csv(OUT / 'scratch_probe_summary.csv', index=False)
    display(summary_df)


## 5. Tổng hợp ảnh khó và diễn giải

Ưu tiên ảnh Validation sai ở cả hai seed, CE cao, dự đoán tự tin hoặc thuộc cặp gần trùng khác nhãn. Hãy mở ảnh gốc, đối chiếu định nghĩa lớp và nhờ người biết bệnh lá xác nhận. Nếu mô hình dao động mạnh nhưng danh sách lỗi không lặp, cần kiểm tra pipeline/BN/LR thêm trước khi quy kết cho dữ liệu.


In [ ]:
if RUN_SCRATCH_PROBE:
    combined = pd.concat(all_val_predictions, ignore_index=True)
    combined['wrong_confidence'] = np.where(~combined.correct, combined.predicted_confidence, np.nan)
    queue = combined.groupby(['relative_path', 'class'], as_index=False).agg(
        errors=('correct', lambda v: int((~v).sum())),
        mean_cross_entropy=('cross_entropy', 'mean'),
        max_wrong_confidence=('wrong_confidence', 'max'),
        predicted_classes=('prediction', lambda v: '|'.join(map(str, v))))
    near_paths = set(near_df.a) | set(near_df.b)
    queue['near_duplicate_candidate'] = queue.relative_path.isin(near_paths)
    queue = queue.sort_values(['errors', 'mean_cross_entropy'], ascending=False)
    queue.to_csv(OUT / 'validation_review_queue.csv', index=False)
    display(queue.head(30))
    contact_sheet(queue.head(90).relative_path, OUT / 'validation_error_contact_sheet.png',
                  'Validation images with repeated errors / high loss')
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for seed in SEEDS:
        curve = pd.read_csv(OUT / f'seed_{seed}' / 'learning_curve.csv')
        ax[0].plot(curve.epoch, curve.train_clean_probe_accuracy, label=f'{seed} Train clean subset')
        ax[0].plot(curve.epoch, curve.val_accuracy, ls='--', label=f'{seed} Val')
        ax[1].plot(curve.epoch, curve.val_loss, label=f'{seed} Val')
    ax[0].set(xlabel='Epoch', ylabel='Accuracy', title='Inference-mode Train vs Validation')
    ax[1].set(xlabel='Epoch', ylabel='Loss', title='Validation loss')
    for a in ax: a.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(OUT / 'scratch_probe_curves.png', dpi=150); plt.show()
    print('Repeated Validation errors:', int((queue.errors == len(SEEDS)).sum()))
print('Audit files:', OUT)
